## Playbook 2 — out of memory

**Symptom:** processes killed unexpectedly, "Killed" in `dmesg`, sluggishness, swap thrashing.

**1. Current state:**

```bash
free -h
```

⚠️ Look at **`available`**, not `free` — that's what's actually allocatable. `free` looks small because the page cache counts as "used" but is reclaimable.

**2. What's using it?**

```bash
ps -eo pid,user,rss,comm --sort=-rss | head    # top by real RAM
top      # then press M to sort by memory
```

**`rss`** (resident set size) is the honest "RAM in use" number; **`vsz`** (virtual size) is usually much larger — ignore it unless chasing an address-space leak.

**3. Check the OOM killer's verdict:**

```bash
sudo dmesg -T | grep -i 'killed process\|out of memory'
```

The kernel's OOM killer logs exactly what it killed and why (`Out of memory: Killed process 12345 (java)…`).

**4. Check swap** — often the missing piece (module 07). No swap configured? Add a swap file at runtime to buy breathing room:

```bash
sudo fallocate -l 2G /swapfile && sudo chmod 600 /swapfile
sudo mkswap /swapfile && sudo swapon /swapfile
```

**5. Protect a critical service** from the OOM killer via its unit — `OOMScoreAdjust=-500` (−1000 = never killed). The real fix is usually more RAM or fixing the leaking process.
